# Barclays Banking — Cleaned Data

This notebook creates cleaned versions of **Customers, Accounts, Transactions and Branches**.

Cleaning is conservative: exact duplicate rows are removed, IDs are kept unique, text is trimmed, dates are standardized, and invalid foreign-key records are flagged/excluded only where they cannot be related to the parent table.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('Raw_Data')
OUT_DIR = Path('cleaned_data')
OUT_DIR.mkdir(exist_ok=True)

customers = pd.read_csv(DATA_DIR/'customers.csv')
accounts = pd.read_csv(DATA_DIR/'accounts.csv')
transactions = pd.read_csv(DATA_DIR/'transactions.csv')
branches = pd.read_csv(DATA_DIR/'branches.csv')


## 1. Remove exact duplicate rows

In [2]:
datasets = {
    'Customers': customers,
    'Accounts': accounts,
    'Transactions': transactions,
    'Branches': branches
}

for name in datasets:
    before = len(datasets[name])
    datasets[name] = datasets[name].drop_duplicates().copy()
    print(f'{name}: removed {before - len(datasets[name])} exact duplicate rows')

customers, accounts, transactions, branches = [datasets[x] for x in ['Customers','Accounts','Transactions','Branches']]


Customers: removed 11 exact duplicate rows
Accounts: removed 16 exact duplicate rows
Transactions: removed 500 exact duplicate rows
Branches: removed 0 exact duplicate rows


## 2. Clean text fields

In [3]:
for df in [customers, accounts, transactions, branches]:
    text_cols = df.select_dtypes(include=['object', 'string']).columns

    for col in text_cols:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )


## 3. Standardize date columns

In [4]:
for df, cols in [
    (customers, ['DateOfBirth']),
    (accounts, ['OpeningDate']),
    (transactions, ['TransactionDate'])
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = df[col].dt.strftime('%Y-%m-%d')


## 4. Remove duplicate IDs conservatively

In [5]:
pk_map = {
    'CustomerID': customers,
    'AccountID': accounts,
    'TransactionID': transactions,
    'BranchID': branches
}

for pk, df in pk_map.items():
    if pk in df.columns:
        before = len(df)
        df.drop_duplicates(subset=[pk], keep='first', inplace=True)
        print(f'{pk}: removed {before - len(df)} duplicate-ID rows')


CustomerID: removed 0 duplicate-ID rows
AccountID: removed 0 duplicate-ID rows
TransactionID: removed 0 duplicate-ID rows
BranchID: removed 0 duplicate-ID rows


## 5. Validate relationships

In [6]:
# Remove records with invalid parent references. Missing foreign keys are kept for review.
if {'CustomerID'} <= set(accounts.columns) and 'CustomerID' in customers.columns:
    accounts = accounts[accounts['CustomerID'].isna() | accounts['CustomerID'].isin(customers['CustomerID'])].copy()

if {'BranchID'} <= set(accounts.columns) and 'BranchID' in branches.columns:
    accounts = accounts[accounts['BranchID'].isna() | accounts['BranchID'].isin(branches['BranchID'])].copy()

if 'AccountOriginID' in transactions.columns:
    transactions = transactions[transactions['AccountOriginID'].isna() | transactions['AccountOriginID'].isin(accounts['AccountID'])].copy()

if 'AccountDestinationID' in transactions.columns:
    transactions = transactions[transactions['AccountDestinationID'].isna() | transactions['AccountDestinationID'].isin(accounts['AccountID'])].copy()

if 'BranchID' in transactions.columns:
    transactions = transactions[transactions['BranchID'].isna() | transactions['BranchID'].isin(branches['BranchID'])].copy()


## 6. Save cleaned datasets

In [7]:
cleaned = {
    'Customers': customers,
    'Accounts': accounts,
    'Transactions': transactions,
    'Branches': branches
}

for name, df in cleaned.items():
    df.to_csv(OUT_DIR/f'{name}_cleaned.csv', index=False)
    print(f'{name}: {len(df):,} rows saved')


Customers: 1,100 rows saved
Accounts: 1,651 rows saved
Transactions: 49,500 rows saved
Branches: 50 rows saved


## Important note

Negative account balances and missing business values are **not automatically deleted**. They may be valid banking situations. We will flag and investigate them separately in the data-quality issue log.